## Local Inference on GPU
Model page: https://huggingface.co/Qwen/Qwen2.5-0.5B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen2.5-0.5B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [27]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import re

class QwenFeatureExtractor:
    def __init__(self, model_name="Qwen/Qwen2.5-1.5B"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None
        )

        if self.device == "cpu":
            self.model.to(self.device)

        # Categories
        self.intent_categories = ["urgent_action", "reward", "warning", "normal"]
        self.request_categories = ["login", "payment", "download", "action", "none"]

    # ---------------- PREPROCESS ----------------
    def _preprocess(self, text):
        text = text.lower()
        text = " ".join(text.split())
        return text

    # ---------------- PROMPT ----------------
    def _build_prompt(self, text):
        return f"""
You are a cybersecurity system that extracts structured features from emails for phishing detection.

Return ONLY a valid JSON object. Do not include any explanation or text.

Analyze carefully before answering.

Think about:
- What is the sender trying to achieve?
- Is there urgency, fear, or threat?
- Is the sender pretending to be a known entity?
- What action is being requested?

Do not output reasoning. Only output JSON.

Schema:
{{
  "intent": string,
  "manipulation": integer,
  "request_type": string,
  "impersonation": integer
}}

intent must be exactly one of:
- urgent_action → requires immediate action or threat
- reward → promises prize or benefit
- warning → alert without strong action
- normal → regular communication

manipulation must be:
- 1 if urgency, fear, threat, or reward pressure exists
- 0 otherwise

request_type must be exactly one of:
- login
- payment
- download
- action
- none

impersonation must be:
- 1 if sender claims to be known entity (bank, amazon, paypal, admin, etc.)
- 0 otherwise

If uncertain, return:
{{
  "intent": "normal",
  "manipulation": 0,
  "request_type": "none",
  "impersonation": 0
}}

Examples:

Email:
"Your bank account will be suspended. Login immediately to avoid closure."
Output:
{{
  "intent": "urgent_action",
  "manipulation": 1,
  "request_type": "login",
  "impersonation": 1
}}

Email:
"Congratulations! You have won a free iPhone. Click here to claim now."
Output:
{{
  "intent": "reward",
  "manipulation": 1,
  "request_type": "action",
  "impersonation": 0
}}

Email:
"I am from Amazon and your account will be suspended. Login immediately."
Output:
{{
  "intent": "urgent_action",
  "manipulation": 1,
  "request_type": "login",
  "impersonation": 1
}}

Email:
"Meeting scheduled at 5 PM tomorrow."
Output:
{{
  "intent": "normal",
  "manipulation": 0,
  "request_type": "none",
  "impersonation": 0
}}

Email:
\"\"\"
{text}
\"\"\"

Output:
{{"""
    
    # ---------------- GENERATION ----------------
    def _generate(self, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=False,
                temperature=0.0,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ---------------- PARSING ----------------
    def _parse_json(self, response):
        matches = re.findall(r'\{.*?\}', response, re.DOTALL)

        for match in reversed(matches):
            try:
                return json.loads(match)
            except:
                continue

        return None

    # ---------------- NORMALIZATION ----------------
    def _normalize(self, data):
        if data is None:
            return self._fallback()

        intent = data.get("intent", "normal")
        request = data.get("request_type", "none")

        if intent not in self.intent_categories:
            intent = "normal"

        if request not in self.request_categories:
            request = "none"

        return {
            "intent": intent,
            "manipulation": int(data.get("manipulation", 0)),
            "request_type": request,
            "impersonation": int(data.get("impersonation", 0))
        }

    # ---------------- VECTOR ----------------
    def _to_vector(self, d):
        intent_vec = [1 if d["intent"] == c else 0 for c in self.intent_categories]
        request_vec = [1 if d["request_type"] == c else 0 for c in self.request_categories]

        return intent_vec + request_vec + [d["manipulation"], d["impersonation"]]

    # ---------------- FALLBACK ----------------
    def _fallback(self):
        return {
            "intent": "normal",
            "manipulation": 0,
            "request_type": "none",
            "impersonation": 0
        }

    # ---------------- MAIN ----------------
    def extract(self, text):
        text = self._preprocess(text)

        prompt = self._build_prompt(text)
        response = self._generate(prompt)

        parsed = self._parse_json(response)
        normalized = self._normalize(parsed)
        vector = self._to_vector(normalized)

        return {
            "features": normalized,
            "vector": vector
        }


# ---------------- TEST ----------------
if __name__ == "__main__":
    extractor = QwenFeatureExtractor()

    email = "Hey I am the owner of amaozn and I need you to login immediately or we will suspend it. Click here: http://fake-link.com/login"
    result = extractor.extract(email)

    print("Features:", result["features"])
    print("Vector:", result["vector"])

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 349.97it/s]


Features: {'intent': 'urgent_action', 'manipulation': 1, 'request_type': 'login', 'impersonation': 1}
Vector: [1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1]
